In [0]:
import json
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, IntegerType, StringType, StructField, StructType, TimestampType


dbutils.widgets.text("eic_project_id", "")
dbutils.widgets.text("salesforce_opportunity_id", "")
dbutils.widgets.dropdown("decision", "", ["", "match", "not_match"])
dbutils.widgets.text("evidence", "")
dbutils.widgets.dropdown("confirm_write", "no", ["no", "yes"])
project_text = dbutils.widgets.get("eic_project_id").strip()
opportunity_id = dbutils.widgets.get("salesforce_opportunity_id").strip()
if not project_text.isdigit() or not opportunity_id:
    raise ValueError("Enter an EIC project ID and Salesforce opportunity ID from the review queue")
project_id = int(project_text)
pair = (spark.table("automation_s.og.pair_label_queue_current")
        .where((F.col("eic_project_id") == project_id)
               & (F.col("salesforce_opportunity_id") == opportunity_id))
        .limit(1).collect())
if len(pair) != 1:
    raise ValueError("Pair is not in the current unlabelled business-review sample")
print(json.dumps(pair[0].asDict(recursive=True), default=str, ensure_ascii=False))
if dbutils.widgets.get("confirm_write") != "yes":
    dbutils.notebook.exit("Preview only: no label written")
decision = dbutils.widgets.get("decision")
evidence = dbutils.widgets.get("evidence").strip()
if decision not in ("match", "not_match") or len(evidence) < 10:
    raise ValueError("Choose match/not_match and provide at least 10 characters of evidence")
reviewer = spark.sql("SELECT current_user() AS reviewer").first()["reviewer"]
if not reviewer:
    raise ValueError("Cannot identify the reviewer")
schema = StructType([
    StructField("eic_project_id", IntegerType(), False),
    StructField("salesforce_opportunity_id", StringType(), False),
    StructField("is_match", BooleanType(), False),
    StructField("reviewer", StringType(), False),
    StructField("reviewed_at_utc", TimestampType(), False),
    StructField("evidence", StringType(), False),
])
record = [(project_id, opportunity_id, decision == "match", reviewer,
           datetime.now(timezone.utc), evidence)]
spark.createDataFrame(record, schema).write.format("delta").mode("append").saveAsTable(
    "automation_s.og.pair_labels")
print(json.dumps({"status": "labelled", "eic_project_id": project_id,
                  "salesforce_opportunity_id": opportunity_id, "decision": decision,
                  "reviewer": reviewer}))